In [1]:
import pandas as pd
import swcol as sc
import plotly.express as px
import numpy as np

In [2]:
dema_path = '../../data/XM-API/variable_query/2022-12-01_2023-11-30/'

GeneReal = sc.loads.add_tech(dema_path)
gen_by_tech = sc.loads.melt_dates(GeneReal)

Shape Gene_Recurso: (82232, 26)
Shape LitadoRecursos_Sistema: (762, 3)
Shape Gene_Recurso (Melted): (1623408, 4)
Shape Gene_Recurso (Melted, Group by Tech): (110046, 4)


In [3]:
# Definimos las condiciones y los valores correspondientes
conditions = [
    (gen_by_tech['Technology'] == 'HIDRAULICA') & (gen_by_tech['Tech_Type'] != 'FILO DE AGUA') & (gen_by_tech['Tech_Type'] != 'FILO AGUA ESPECIAL'),
    (gen_by_tech['Technology'] == 'TERMICA') | (gen_by_tech['Technology'] == 'COGENERADOR'),
    (gen_by_tech['Technology'] == 'SOLAR'),
    (gen_by_tech['Technology'] == 'EOLICA'),
    (gen_by_tech['Technology'] == 'HIDRAULICA') & ((gen_by_tech['Tech_Type'] == 'FILO DE AGUA') | (gen_by_tech['Tech_Type'] == 'FILO AGUA ESPECIAL')),
]

choices = ["Hydro", "Thermal", "Solar", "Wind", "Run of River"]

# Crear la nueva columna
gen_by_tech['Tech'] = np.select(conditions, choices, default='Other')

gen_by_tech = gen_by_tech.groupby(['Tech','Date']).agg({
    'GeneReal': 'sum'}).reset_index()

print(gen_by_tech.shape)
sc.loads.plot_multiple_line(
    gen_by_tech, 'Date', 'Date', 'GeneReal', 'Tech', 'Generación por tecnología Daily', {'GWh':'GeneReal'})
gen_by_tech.sample(5)

(39888, 3)


,Tech,Date,GeneReal
34228,Wind,2023-03-24 22:00:00,26.00975
26594,Thermal,2023-05-09 16:00:00,2108.18071
32971,Wind,2023-01-31 01:00:00,20.73114
1432,Hydro,2023-01-29 16:00:00,6362.52088
13555,Run of River,2023-06-18 19:00:00,362.91348


In [ ]:
demand_clusterized = sc.loads.cluster(gen_by_tech)
demand_clusterized.to_csv(dema_path+'Melted_Gen_Res.csv', index=False)
demand_clusterized.sample(5)

Shape Gene_Recurso (Melted, Group by Tech, Timestamp): (929, 3)


,timestamp,Tech,GeneReal
347,2023_Q2_labor_10h,Hydro,7254.470339
927,2023_Q4_labor_9h,Thermal,2911.426211
354,2023_Q2_labor_11h,Solar,271.396576
225,2023_Q1_labor_9h,Solar,196.936748
1,2023_Q1_holidays_0h,Run of River,379.196028
